In [7]:
from GradientGang.Pipeline.Pipeline import Pipeline
import pytorch_lightning as L
from GradientGang.Pipeline.Architectures.LightningAutoencoder import LightningAutoencoder

In [8]:
data_params_s = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.1,
    'shuffle': True,
}

In [9]:
pipeline = Pipeline(data_params_s)

In [ ]:
data_params_d = {"stage": "fit", "includeTestInTrain": False}
architectureParams = {
    "arch_type": "autoencoder_joint",
    "LearningRate": 0.001,
    "Patience": 5,
    "RegularizationWeight": 0.1,
    "ReconstructionLossWeight": 0.1,
    "EncoderParams": {
        "activation_function": "GELU",
        "layer_type": [
            {
                "name": "LSTM",
                "params": {
                    "input_size": 34,
                    "hidden_size": 64,
                    "num_layers": 1,
                    "bias": True,
                    "batch_first": True,
                    "dropout": 0.0,
                    "bidirectional": False
                }
            },
            {
                "name": "Linear",
                "params": {
                    "in_features": 64,  # LSTM hidden_size
                    "out_features": 128,
                    "bias": True,
                    "device": None,
                    "dtype": None
                }
            }
        ]
    },
    "DecoderParams": {
        "activation_function": "GELU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 128,
                    "out_features": 64,  # Changed to match LSTM input_size
                    "bias": True,
                    "device": None,
                    "dtype": None
                }
            },
            {
                "name": "LSTM",
                "params": {
                    "input_size": 64,  # This will receive (batch, seq_len, 64)
                    "hidden_size": 34,
                    "num_layers": 1,
                    "bias": True,
                    "batch_first": True,
                    "dropout": 0.0,
                    "bidirectional": False
                }
            }
        ]
    },
    "FeedForwardParams": {
        "activation_function": "ReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 129,
                    "out_features": 64,
                    "bias": True,
                    "device": None,
                    "dtype": None
                }
            },
            {
                "name": "Linear",
                "params": {
                    "in_features": 64,
                    "out_features": 10,
                    "bias": True,
                    "device": None,
                    "dtype": None
                }
            }
        ]
    },
    "GlobalFFEncoderParams": {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 1,
                    "out_features": 1,
                    "bias": True,
                    "device": None,
                    "dtype": None
                }
            },
        ]
    },
    "GlobalFFDecoderParams": {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": 1,
                    "out_features": 1,
                    "bias": True,
                    "device": None,
                    "dtype": None
                }
            },
        ]
    },
    "OutputDim": 3
}

In [11]:
arch = pipeline.build_architecture(architectureParams)

In [12]:
pipeline.fit_and_validate(architectureParams, data_params_d)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\loops\utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.

  | Name             | Type              | Params | Mode 
---------------------------------------------------------------
0 | encoder          | Encoder           | 33.9 K | train
1 | decoder          | Decoder           | 21.9 K | train
2 | globalff_encoder | FeedForward       | 2      | train
3 | globalff_decoder | FeedForward       | 2      | train
4 | feedforward      | FeedForward       | 9.0 K  | train
5 | val_f1           | MulticlassF1Score | 0      | train
---------------------------------------------------------------
64.8 K    Trainable params
0         Non-trainable params
64.8 K    Total params
0.259     Total estimated model 

c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\loren\Desktop\School\Master\AN2DL\Challenges\ANN-Challenges\.venv\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (19) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to s

Epoch 20: 100%|██████████| 19/19 [00:00<00:00, 24.93it/s, v_num=6, val_reconstruction_loss=0.658, val_F1=0.509]



tensor(0.5876)